In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# ── Approach: Two clean files ───────────────────────────────────────────────

# File 1: Standardised — KG and Litres only
# These are your spike detection and choropleth data
df_standard = df[df['unit'].isin(['KG', 'L'])].copy()

# Fix the one real problem — Groundnuts outlier
outlier_mask = (
    (df_standard['commodity'] == 'Groundnuts (shelled)') &
    (df_standard['price'] > 25000)
)
print(f'Groundnuts outliers removed: {outlier_mask.sum()} rows')
df_standard = df_standard[~outlier_mask]

# File 2: Informal — Heap, Bunch, Unit kept exactly as collected
df_informal = df[df['unit'].isin(['Heap', 'Bunch', 'Unit'])].copy()

# Add price index per commodity (% change from first observation)
# This lets you track trends WITHOUT converting units
df_informal = df_informal.sort_values(
    ['district', 'market', 'commodity', 'date']
).reset_index(drop=True)

df_informal['price_pct_change'] = (
    df_informal
    .groupby(['district', 'market', 'commodity'])['price']
    .pct_change() * 100
)

# ── Summary ───────────────────────────────────────────────────────────────────
print('=== FILE 1: Standardised (KG + L) ===')
print(f'Records     : {len(df_standard):,}')
print(f'Commodities : {df_standard["commodity"].nunique()}')
print(sorted(df_standard["commodity"].unique()))

print('\n=== FILE 2: Informal (Heap + Bunch + Unit) ===')
print(f'Records     : {len(df_informal):,}')
print(f'Commodities : {df_informal["commodity"].nunique()}')
print(sorted(df_informal["commodity"].unique()))

# ── Save ──────────────────────────────────────────────────────────────────────
df_standard.to_csv('food_prices_standardised.csv', index=False)
df_informal.to_csv('food_prices_informal.csv', index=False)

print('\n✅ food_prices_standardised.csv → use for spike detection')
print('✅ food_prices_informal.csv     → use for informal market layer')

In [ ]:
# the Critical spikes
critical = df[df["spike_severity"] == "Critical"].copy()

print('=== CRITICAL SPIKES BREAKDOWN ===\n')
print('By Region:')
print(critical['region'].value_counts())

print('\nBy District top10:')
print(critical['district'].value_counts().head(10))

print('\nBy Commodity top10:')
print(critical['commodity'].value_counts().head(10))

print('\nBy Year:')
print(critical['date'].value_counts().sort_index())

print('\nWorst single events (highest pct_change):')
print(critical.nlargest(10, 'pct_change')[
    ['date','district','market','commodity','price','pct_change','zscore']
].to_string(index=False))

In [ ]:
# District risk summary — this becomes choropleth colour data
district_risk = (
    df.groupby(['region', 'district'])
    .agg(
        total_records    = ('price', 'count'),
        total_spikes     = ('is_spike', 'sum'),
        critical_count   = ('spike_severity', lambda x: (x == 'Critical').sum()),
        severe_count     = ('spike_severity', lambda x: (x == 'Severe').sum()),
        moderate_count   = ('spike_severity', lambda x: (x == 'Moderate').sum()),
        avg_price        = ('price', 'mean'),
        avg_pct_change   = ('pct_change', 'mean'),
        worst_spike      = ('pct_change', 'max'),
    )
    .reset_index()
)

# Spike rate per district — % of months that had any spike
district_risk['spike_rate_pct'] = (
    district_risk['total_spikes'] / district_risk['total_records'] * 100
).round(2)

# Risk score — weighted composite
# Critical counts 3x, Severe 2x, Moderate 1x
district_risk['risk_score'] = (
    district_risk['critical_count'] * 3 +
    district_risk['severe_count']   * 2 +
    district_risk['moderate_count'] * 1
)

# Rank districts by risk
district_risk = district_risk.sort_values(
    'risk_score', ascending=False
).reset_index(drop=True)

district_risk['risk_rank'] = district_risk.index + 1

print('=== DISTRICT RISK RANKING ===\n')
print(district_risk[[
    'risk_rank','region','district',
    'critical_count','severe_count','moderate_count',
    'spike_rate_pct','risk_score'
]].to_string(index=False))

In [ ]:
# ── Visualise: Top 10 most volatile districts ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Malawi Food Price Spike Analysis by District', fontsize=14, fontweight='bold')

spike_by_district = df[df['spike_severity'] != 'Normal'] \
    .groupby(['district', 'region']) \
    .agg(
        total_spikes=('spike_severity', 'count'),
        avg_pct_change=('pct_change', 'mean')
    ) \
    .reset_index() \
    .sort_values(by='total_spikes', ascending=False)

top10 = spike_by_district.head(10)

# Bar chart: spike count
colors = ['#d32f2f' if r == 'Southern Region' else '#1565c0' if r == 'Central Region' else '#2e7d32'
          for r in top10['region']]
axes[0].barh(top10['district'], top10['total_spikes'], color=colors)
axes[0].set_xlabel('Number of Spike Events')
axes[0].set_title('Total Price Spike Events')
axes[0].invert_yaxis()

# Bar chart: avg pct change during spikes
axes[1].barh(top10['district'], top10['avg_pct_change'], color=colors)
axes[1].set_xlabel('Average % Price Jump')
axes[1].set_title('Average % Change During Spikes')
axes[1].invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend = [
    Patch(color='#d32f2f', label='Southern Region'),
    Patch(color='#1565c0', label='Central Region'),
    Patch(color='#2e7d32', label='Northern Region'),
]
axes[0].legend(handles=legend, loc='lower right')

plt.tight_layout()
plt.savefig('spike_by_district.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as spike_by_district.png')

In [ ]:
# ── Visualise: Price timeline for one commodity across regions ─────────────────
# Ensure date is datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# ── Visualise: Price timeline for maize across regions ─────────────────
commodity_focus = 'maize'  # use flexible matching

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
fig.suptitle('Price Timeline: Maize (All Types) by Region', fontsize=13, fontweight='bold')

regions = ['Southern Region', 'Central Region', 'Northern Region']
region_colors = ['#d32f2f', '#1565c0', '#2e7d32']

# Filter spike events
spikes = df[df['spike_severity'] != 'Normal'].copy()

for ax, region, color in zip(axes, regions, region_colors):
    
    # Filter maize-related data (case-insensitive)
    region_data = df[
        df['commodity'].str.contains(commodity_focus, case=False, na=False) &
        (df['region'] == region)
    ].groupby('date')['price'].median().reset_index()
    
    region_spikes = spikes[
        spikes['commodity'].str.contains(commodity_focus, case=False, na=False) &
        (spikes['region'] == region)
    ]

    # Plot price trend
    ax.plot(region_data['date'], region_data['price'],
            color=color, linewidth=1.5, label='Median price')

    # Overlay spike events
    if not region_spikes.empty:
        spike_monthly = region_spikes.groupby('date')['price'].median().reset_index()
        ax.scatter(spike_monthly['date'], spike_monthly['price'],
                   color='red', s=25, zorder=5, label='Spike event', alpha=0.7)

    ax.set_title(region, fontsize=11)
    ax.set_ylabel('Price (MWK)')
    ax.legend(loc='upper left', fontsize=8)
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('maize_price_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

print('Chart saved as maize_price_timeline.png')

In [ ]:
# The +838% Beans spike in Mchinji needs investigation
# before it goes on map and misleads a decision maker

df['date'] = pd.to_datetime(df['date'])

suspicious = df[df['pct_change'] > 400].copy()
print('=== EVENTS ABOVE 400% — VERIFY BEFORE MAPPING ===\n')
print(suspicious[[
    'date','region','district','market',
    'commodity','price','pct_change','zscore'
]].to_string(index=False))
print(f'\nTotal suspicious events: {len(suspicious)}')

# Check what the price was BEFORE each suspicious spike
# to understand if it looks like a data error
for idx, row in suspicious.iterrows():
    prev = df[
        (df['district']  == row['district']) &
        (df['market']    == row['market']) &
        (df['commodity'] == row['commodity']) &
        (df['date']      < row['date'])
    ].tail(3)
    print(f"\n{row['market']} — {row['commodity']} — {row['date'].date()}")
    print(f"Spike price    : {row['price']:,.0f} MWK")
    print(f"Previous prices:")
    print(prev[['date','price']].to_string(index=False))

In [ ]:
# Save district risk ranking
district_risk.to_csv('malawi_district_risk.csv', index=False)
print('Saved: malawi_district_risk.csv')
print('   → This feeds choropleth map colours in Phase 5')

# Save spike events only
spikes = df[df['spike_severity'] != 'Normal'].copy()
spikes.to_csv('malawi_spikes_all.csv', index=False)
print(f'Saved: malawi_spikes_all.csv ({len(spikes):,} spike events)')

# Save critical only — your priority alert layer
critical = df[df['spike_severity'] == 'Critical'].copy()
critical.to_csv('malawi_spikes_critical.csv', index=False)
print(f'Saved: malawi_spikes_critical.csv ({len(critical):,} critical events)')

# Save full clean dataset with all new columns
df.to_csv('food_prices_analysed.csv', index=False)
print(f'Saved: food_prices_analysed.csv (complete dataset with spike flags)')

print('\n=== PHASE 1 COMPLETE — FILES PRODUCED ===')
print('''
food_prices_standardised.csv   → cleaned input data
food_prices_informal.csv       → informal market data  
malawi_district_risk.csv       → choropleth input (Phase 2 QGIS)
malawi_spikes_all.csv          → all spike events
malawi_spikes_critical.csv     → critical alerts only
food_prices_analysed.csv       → complete dataset with all flags
''')